# 01 · Prepare the ISEAR stories

In [15]:
import json, math, re
from pathlib import Path
import pandas as pd

## Settings and data download

In [16]:
# pip install huggingface_hub pyarrow
from huggingface_hub import hf_hub_download
OUT = Path("data"); OUT.mkdir(exist_ok=True)
ISEAR_PATH = hf_hub_download(repo_id="savalera/isear-from-original",
                             filename="original/train-00000-of-00001.parquet",
                             repo_type="dataset")
print("Downloaded:", ISEAR_PATH)
N_COUNTRIES = 2                        # pre-registered: the two countries with most valid stories
CUE_SAMPLE_PER_COUNTRY = 60
SEED = 2026

Downloaded: C:\Users\maaye\.cache\huggingface\hub\datasets--savalera--isear-from-original\snapshots\9741e2c3fc20757958c9b67ce61aa09016be6cd8\original\train-00000-of-00001.parquet


## Codebook, exclusion rule and cue dictionary

In [17]:
COUNTRIES = {
    "1": "Sweden", "2": "Norway", "3": "West Germany", "4": "Finland", "5": "Greece",
    "6": "Hong Kong", "7": "Lebanon", "8": "Austria", "9": "Australia", "10": "Brazil",
    "11": "Botswana", "12": "Bulgaria", "13": "France", "14": "Italy", "15": "Japan",
    "16": "New Zealand", "17": "the Netherlands", "18": "Portugal", "19": "Spain",
    "20": "Zambia", "21": "Zimbabwe", "22": "the United States", "23": "Poland",
    "24": "Nigeria", "25": "Israel", "26": "India", "27": "Malawi", "28": "Switzerland",
    "29": "Chile", "30": "China", "31": "Yugoslavia", "32": "Costa Rica", "33": "Honduras",
    "34": "Mexico", "35": "Guatemala", "36": "Venezuela", "37": "El Salvador",
}
SEX = {"1": "man", "2": "woman"}                      # 0 = missing
FIELD = {"1": "psychology", "2": "the social sciences", "3": "languages",
         "4": "fine arts", "5": "law", "6": "natural sciences",
         "7": "engineering", "8": "medicine", "9": "another subject"}  # 0 = missing
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]

INVALID = re.compile(
    r"^\[?\s*(?:nothing|none|no response|no answer|no description|not applicable|"
    r"not included|blank|n/?a\b|do not (know|remember|recall)|can'?t remember|"
    r"cannot (remember|recall)|can not (remember|recall|think)|never|i (have )?never|"
    r"i can ?not (recall|remember|think)|i do not (recall|remember)|same as|the same|as for|"
    r"doesn'?t apply)", re.I)

# Explicit nationality cues for the countries in the MVP. Extend before running.
CUES = {
    "Sweden": ["swed\\w*", "stockholm", "uppsala", "gothenburg", "g[oö]teborg", "malm[oö]",
               "lund", "ume[aå]", "krona", "kronor", "scandinavia\\w*", "nordic"],
    "Zambia": ["zambia\\w*", "lusaka", "kitwe", "ndola", "kabwe", "livingstone", "copperbelt",
               "kwacha", "ngwee", "bemba", "nyanja", "lozi", "tonga", "chewa", "nshima",
               "chitenge", "africa\\w*"],
    "China": ["china", "chinese", "beijing", "peking", "shanghai", "canton\\w*", "guangzhou",
              "yuan", "renminbi", "rmb", "mao", "communist party", "cultural revolution",
              "spring festival", "lunar new year", "taiwan\\w*", "hong kong", "korea\\w*",
              "japan\\w*", "opium war", "asia\\w*"],
    "_any": ["europe\\w*", "america\\w*", "english", "british"],
}


## Loading function

In [18]:
def load_isear(path):
    """Load ISEAR from parquet or CSV and return every column as a clean string.
    Works for the Hugging Face parquet (savalera/isear-from-original) and pipe/comma CSVs."""
    path = str(path)
    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            sep = "|" if "|" in f.readline() else ","
        df = pd.read_csv(path, sep=sep, dtype=str, on_bad_lines="skip")
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]) or pd.api.types.is_integer_dtype(df[c]):
            df[c] = df[c].astype("Int64")
        df[c] = df[c].astype("string").fillna("").str.strip()
    if "Field1" not in df.columns:            # emotion as a word
        df["Field1"] = df["EMOT"]
    if "MYKEY" not in df.columns:
        df["MYKEY"] = ""
    return df


## Power function

In [19]:
def mde(n_stories, n_writers, discordance, icc, alpha_z=1.645, power_z=0.842):
    """Minimum detectable difference for a paired binary outcome, one-sided test.

    discordance = share of stories whose correctness differs between the two conditions.
    icc         = correlation of outcomes between stories of the same writer.
    """
    m = n_stories / n_writers
    design_effect = 1 + (m - 1) * icc
    n_eff = n_stories / design_effect
    return (alpha_z + power_z) * math.sqrt(discordance / n_eff)


## 1. Load and check the data against the project sheet

In [20]:
raw = load_isear(ISEAR_PATH)
raw.to_csv(OUT / "isear_original.csv", index=False)
print(raw.shape)
display(raw[(raw.ID == "201072") & (raw.Field1.str.lower() == "guilt")][
    ["ID", "COUN", "SUBJ", "SEX", "AGE", "FIEL", "Field1", "MYKEY", "SIT"]])
for col in ["COUN", "SEX", "FIEL", "Field1"]:
    print(col, raw[col].value_counts().sort_index().to_dict())

(6132, 42)


,ID,COUN,SUBJ,SEX,AGE,FIEL,Field1,MYKEY,SIT
815,201072,20,72,2,19,5,guilt,2010727,When my friend was punished for something that...


COUN {'1': 451, '10': 319, '12': 411, '16': 334, '17': 395, '19': 439, '2': 204, '20': 621, '22': 336, '26': 379, '27': 409, '30': 431, '33': 306, '4': 388, '8': 388, '9': 321}
SEX {'0': 4, '1': 2757, '2': 3371}
FIEL {'0': 64, '1': 2527, '2': 888, '3': 361, '4': 61, '5': 267, '6': 264, '7': 83, '8': 1042, '9': 575}
Field1 {'anger': 877, 'disgust': 871, 'fear': 882, 'guilt': 876, 'joy': 877, 'sadness': 878, 'shame': 871}


## 2. Clean

In [21]:
df = raw[raw.COUN.isin(COUNTRIES)].copy()
df["country"] = df.COUN.map(COUNTRIES)
df["writer_id"] = df.COUN + "_" + df.SUBJ
df["story_id"] = df.MYKEY.where(df.MYKEY != "", df.ID + "_" + df.EMOT)
df["emotion"] = df.Field1.str.strip().str.lower()
df["text"] = (df.SIT.fillna("").str.replace("á", "", regex=False)
              .str.replace(r"\s+", " ", regex=True).str.strip())
df["invalid"] = df.text.str.len().lt(5) | df.text.str.contains(INVALID)
df = df[df.emotion.isin(EMOTIONS)]
print(f"Loaded {len(df)} stories; removed {df.invalid.sum()} invalid answers.")
df[df.invalid].text.head(10).tolist()

Loaded 6132 stories; removed 132 invalid answers.


C:\Users\maaye\AppData\Local\Temp\ipykernel_8616\975836605.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["invalid"] = df.text.str.len().lt(5) | df.text.str.contains(INVALID)


['No response.',
 'No response.',
 'Do not remember any incident.',
 'Cannot remember such a situation.',
 'The same situation as with sadness. I was afraid about what my parents would further do to distroy my relationship.',
 'No response.',
 'No response.',
 'I can not think of any incident when I have felt very ashamed. There are small incidents that I am ashamed of, for instance, not sending money to the children in Ethiopia.',
 'No response.',
 'No response.']

## 3. Choose the two countries

In [22]:
valid = df[~df.invalid].copy()
counts = valid.groupby("country").size().sort_values(ascending=False)
chosen = list(counts.index[:N_COUNTRIES])
print("Chosen countries:", chosen)
counts.to_frame("valid stories")

Chosen countries: ['Zambia', 'Spain']


,valid stories
country,
Zambia,621
Spain,435
China,431
Bulgaria,411
Malawi,407
Sweden,398
the Netherlands,391
Austria,388
Finland,386


## 4. Persona attributes and nationality cues

In [23]:
s = valid[valid.country.isin(chosen)].copy()
s["sex"] = s.SEX.map(SEX)
s["age"] = pd.to_numeric(s.AGE, errors="coerce")
s["field"] = s.FIEL.map(FIELD)
s["full_persona_complete"] = s[["sex", "age", "field"]].notna().all(axis=1)

pattern = {c: re.compile(r"\b(?:" + "|".join(CUES[c] + CUES["_any"]) + r")\b", re.I)
           for c in chosen if c in CUES}
missing = [c for c in chosen if c not in CUES]
if missing:
    print("WARNING: add cue words for", missing)
s["cue_flag"] = [bool(pattern[c].search(t)) if c in pattern else False
                 for c, t in zip(s.country, s.text)]

display(pd.crosstab(s.country, s.emotion))
display(s.groupby("country").agg(stories=("story_id", "size"), writers=("writer_id", "nunique"),
                                 cue_share=("cue_flag", "mean"), mean_age=("age", "mean")))

emotion,anger,disgust,fear,guilt,joy,sadness,shame
country,,,,,,,
Spain,63,64,63,63,61,62,59
Zambia,90,87,86,94,88,86,90


,stories,writers,cue_share,mean_age
country,,,,
Spain,435,78,0.000000,20.967816
Zambia,621,110,0.022544,22.36876


In [24]:
for c in chosen:
    if c in pattern:
        hits = s[s.country == c].text.map(lambda t: [x.lower() for x in pattern[c].findall(t)])
        print(c, pd.Series([w for l in hits for w in l]).value_counts().head(10).to_dict())

Zambia {'zambia': 8, 'lusaka': 2, 'copperbelt': 1, 'kabwe': 1, 'ndola': 1, 'african': 1}


## 5. Save the story table

In [25]:
cols = ["story_id", "writer_id", "country", "emotion", "text", "sex", "age",
        "field", "full_persona_complete", "cue_flag"]
s[cols].to_csv(OUT / "stories.csv", index=False)
print("Saved", len(s), "stories to", OUT / "stories.csv")

Saved 1056 stories to data\stories.csv


## 6. Blind sample for hand-coding the cue flag

For each story, write 1 in `hand_cue` if a reader could guess the writer's country from the text, otherwise 0. The file deliberately hides the dictionary's own flag.

In [26]:
samp = s.groupby("country").sample(n=CUE_SAMPLE_PER_COUNTRY, random_state=SEED)
samp[["story_id", "country", "text"]].assign(hand_cue="").to_csv(OUT / "validation_cue_sample.csv", index=False)
print("Exported", len(samp), "stories to", OUT / "validation_cue_sample.csv")

Exported 120 stories to data\validation_cue_sample.csv


## 7. Smallest detectable difference

Report this range in the pre-registration, before the experiment is run.

In [27]:
n, w = len(s), s.writer_id.nunique()
power = {f"discordance={d}, icc={icc}": round(100 * mde(n, w, d, icc), 2)
         for d in (0.10, 0.20) for icc in (0.05, 0.10)}
summary = {"source_file": ISEAR_PATH, "countries": chosen,
           "stories_per_country": s.groupby("country").size().to_dict(),
           "writers_per_country": s.groupby("country").writer_id.nunique().to_dict(),
           "stories_total": n, "writers_total": w,
           "full_persona_complete": int(s.full_persona_complete.sum()),
           "cue_flagged": s.groupby("country").cue_flag.sum().to_dict(),
           "mde_percentage_points": power}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=2, default=int))
summary

{'source_file': 'C:\\Users\\maaye\\.cache\\huggingface\\hub\\datasets--savalera--isear-from-original\\snapshots\\9741e2c3fc20757958c9b67ce61aa09016be6cd8\\original\\train-00000-of-00001.parquet',
 'countries': ['Zambia', 'Spain'],
 'stories_per_country': {'Spain': 435, 'Zambia': 621},
 'writers_per_country': {'Spain': 78, 'Zambia': 110},
 'stories_total': 1056,
 'writers_total': 188,
 'full_persona_complete': 1040,
 'cue_flagged': {'Spain': 0, 'Zambia': 14},
 'mde_percentage_points': {'discordance=0.1, icc=0.05': 2.69,
  'discordance=0.1, icc=0.1': 2.93,
  'discordance=0.2, icc=0.05': 3.8,
  'discordance=0.2, icc=0.1': 4.14}}